# ICONCRETE — مولد مصنف Excel المحمول | Portable Excel Workbook Generator

**ICONCRETE — Concrete Management Solutions**

هذا الدفتر يولّد ملف `.xlsx` حقيقياً كاملاً لنظام إدارة محطات الخرسانة الجاهزة، ويعمل بالكامل داخل Google Colab
بدون خادم، بدون مفاتيح API، وبدون Microsoft Excel Desktop، وبدون تعديل يدوي لكود Python.

**طريقة الاستخدام:**
1. ارفع مجلد `excel_generator` إلى Colab (أو استنسخ المستودع) ثم شغّل الخلايا بالترتيب من ١ إلى ٩.
2. خلية تركيب Google Drive اختيارية ومعطّلة افتراضياً.
3. خلية رفع البيانات اختيارية ومعطّلة افتراضياً — المولد يعمل فوراً على البيانات التجريبية الموسومة DEMO.
4. عدّل إعدادات الهوية/الفترة في خلية الإعدادات فقط، ثم أعد التوليد.

**What this notebook does:** generates a real `.xlsx` workbook (Arabic RTL by default) containing the
Login cover, Dashboard with charts, Validation_Report, Data Dictionary, Settings, seven data sheets,
Migration guide and Demo_Data guide — with Excel tables, formulas, dropdown validations, conditional
formatting and protected formula cells.

**It does not:** send data anywhere, call any paid service, execute macros, or claim live cloud sync / RLS /
2FA / GPS / WhatsApp / Power Pivot. Those are documented limitations in the workbook README sheet.


In [ ]:
# ---------------------------------------------------------------------------
# 2) خلية اختيارية: تركيب Google Drive — معطّلة افتراضياً
#    OPTIONAL: mount Google Drive — disabled by default
# ---------------------------------------------------------------------------
USE_GOOGLE_DRIVE = False

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive  # type: ignore
        drive.mount('/content/drive')
        print('Google Drive mounted at /content/drive')
    except Exception as exc:  # running outside Colab
        print('Google Drive mount is only available inside Google Colab:', exc)
else:
    print('Google Drive mount skipped (disabled by default).')


In [ ]:
# ---------------------------------------------------------------------------
# 3) تثبيت المتطلبات | Install dependencies
# ---------------------------------------------------------------------------
import subprocess, sys

def install(packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])

try:
    import openpyxl  # noqa: F401
    print('openpyxl already installed:', openpyxl.__version__)
except ImportError:
    install(['openpyxl>=3.1.2,<4', 'pytest>=8.0'])
    import openpyxl
    print('Installed openpyxl:', openpyxl.__version__)


In [ ]:
# ---------------------------------------------------------------------------
# 4) الإعدادات — القيم المقفلة من Addendum A1 (لا تحتاج تأكيداً)
#    Configuration — locked defaults, edit only what you need
# ---------------------------------------------------------------------------
import os, sys
from datetime import date, timedelta

# عدّل المسار إذا رفعت المجلد في مكان آخر داخل Colab
MODULE_DIR = os.environ.get('ICONCRETE_MODULE_DIR', '/content/excel_generator')
if MODULE_DIR not in sys.path:
    sys.path.insert(0, MODULE_DIR)

# إن كنت تشغّل الدفتر من داخل مجلد excel_generator نفسه
if not os.path.exists(os.path.join(MODULE_DIR, 'config.py')):
    MODULE_DIR = os.getcwd()
    sys.path.insert(0, MODULE_DIR)

from config import GeneratorConfig

CFG = GeneratorConfig(
    # الهوية المقفلة
    company_name_ar='شركة آي كونكريت التجريبية',
    company_name_en='IConcrete Demo Company',
    plant_name_ar='محطة آي كونكريت التجريبية - القاهرة',
    plant_name_en='IConcrete Demo Plant - Cairo',
    currency='EGP',
    language='ar',
    timezone='Africa/Cairo',
    # فترة التقرير
    report_from=date.today() - timedelta(days=150),
    report_to=date.today(),
    # سلوك المولد
    # Colab writes to /content; local runs write to ./output next to this notebook
    output_dir=('/content/iconcrete_output' if os.path.isdir('/content') else os.path.join(os.getcwd(), 'output')),
    protect_sheets=True,
    include_demo_data=True,
    include_charts=True,
    vat_rate=14.0,
)

print('Module dir :', MODULE_DIR)
print('Company    :', CFG.company_name_ar, '/', CFG.company_name_en)
print('Period     :', CFG.report_from, '->', CFG.report_to)
print('Output dir :', CFG.output_dir)


In [ ]:
# ---------------------------------------------------------------------------
# 5) خلية اختيارية: رفع ملف CSV/Excel — معطّلة افتراضياً
#    OPTIONAL: upload your own CSV/XLSX input — disabled by default
# ---------------------------------------------------------------------------
USE_UPLOAD = False

if USE_UPLOAD:
    try:
        from google.colab import files  # type: ignore
        uploaded = files.upload()
        print('Uploaded files:', list(uploaded.keys()))
        print('Demo sample inputs live in excel_generator/sample_input/')
    except Exception as exc:
        print('Upload is only available inside Google Colab:', exc)
else:
    print('Optional upload skipped (disabled by default).')


In [ ]:
# ---------------------------------------------------------------------------
# 6) توليد المصنف | Generate the workbook
# ---------------------------------------------------------------------------
from generate_workbook import generate, EXPECTED_SHEETS

RESULT = generate(CFG, CFG.output_dir)

print('Workbook generated successfully.')
print('Path  :', RESULT['path'])
print('Sheets:', len(RESULT['sheets']))
print('Charts:', RESULT['charts'])


In [ ]:
# ---------------------------------------------------------------------------
# 7) التحقق الآلي والاختبارات | Automated validation and tests
# ---------------------------------------------------------------------------
import os
from openpyxl import load_workbook
import generate_workbook as gen
import config as cfg_module

book = load_workbook(RESULT['path'])
table_map = {
    'Customers': 'tblCustomers',
    'Orders': 'tblOrders',
    'Materials': 'tblMaterials',
    'Mix_Designs': 'tblMixDesigns',
    'Employees': 'tblEmployees',
    'Fleet': 'tblFleet',
    'Suppliers': 'tblSuppliers',
}
scenarios = gen.validate_demo_scenarios(CFG.report_to)

checks = [
    ('output file exists', os.path.exists(RESULT['path'])),
    ('workbook opens with openpyxl', len(book.sheetnames) > 0),
    ('all expected sheets exist', book.sheetnames == list(EXPECTED_SHEETS)),
    ('locked brand text on Login', book['Login']['A1'].value == 'ICONCRETE'),
    ('tagline on Login', book['Login']['A2'].value == 'Concrete Management Solutions'),
    ('RTL on every sheet', all(sheet.sheet_view.rightToLeft for sheet in book.worksheets)),
    ('dashboard charts exist', len(book['Dashboard']._charts) >= 3),
    ('excel tables exist', all(name in book[sheet].tables for sheet, name in table_map.items())),
    ('named ranges exist', 'VAT_Rate' in book.defined_names and 'Company_Name' in book.defined_names),
    ('dropdown validations exist', all(len(book[s].data_validations.dataValidation) > 0 for s in table_map)),
    ('formulas are real formulas', str(book['Customers']['I2'].value).startswith('=')),
    ('formula cells are protected', book['Customers']['I2'].protection.locked is True),
    ('input cells are editable', book['Customers']['A2'].protection.locked is False),
    ('demo data passes validation', gen.evaluate_demo_data_issues(cfg_module.build_demo_data(CFG), CFG.report_to) == []),
    ('seeded error scenarios are detected', all(item['detected'] for item in scenarios)),
]

failed = 0
for name, ok in checks:
    print(('PASS' if ok else 'FAIL'), '-', name)
    if not ok:
        failed += 1

print()
print('RESULT:', len(checks) - failed, '/', len(checks), 'checks passed')
assert failed == 0, 'Some validation checks failed — review the FAIL lines above'


In [ ]:
# ---------------------------------------------------------------------------
# 8) المعاينة والسجل | Preview and log
# ---------------------------------------------------------------------------
print('=' * 60)
print('ICONCRETE — Workbook summary')
print('=' * 60)
print('Workbook path :', RESULT['path'])
print('Sheets        :', ', '.join(RESULT['sheets']))
print('Charts        :', RESULT['charts'])
print('RTL sheets    :', RESULT['rtl_sheets'])
print('File size     :', RESULT['bytes'], 'bytes')
print('Data issues   :', len(RESULT['data_issues']))
print('Seeded tests  :', RESULT['seeded_detected'], '/', RESULT['seeded_scenarios'], 'detected')
status = 'OK' if not RESULT['data_issues'] and RESULT['seeded_detected'] == RESULT['seeded_scenarios'] else 'CHECK REQUIRED'
print('Validation    :', status)
print('=' * 60)


In [ ]:
# ---------------------------------------------------------------------------
# 9) تنزيل الملف إلى جهازك | Download the final .xlsx
# ---------------------------------------------------------------------------
import os, shutil

filename = os.path.basename(RESULT['path'])

try:
    from google.colab import files  # type: ignore
    files.download(RESULT['path'])
    print('Download started:', filename)
except Exception:
    target = os.path.join('/content', filename) if os.path.isdir('/content') else filename
    shutil.copy(RESULT['path'], target)
    print('Colab download unavailable (running locally). File copied to:', target)
